# Bronze — VRA (Voo Regular Ativo)

Lê os 12 CSVs mensais do volume `voo_bem.bronze.arquivos.vra/` e materializa `voo_bem.bronze.vra`.

## Regras da Camada Bronze

- **Nada de tipagem**: todos os dados devem permanecer como `string`, exatamente como vieram do arquivo original.
- **Nada de filtro**: nenhuma linha deve ser descartada durante a ingestão.
- **Colunas de auditoria**: incluir informações sobre a origem dos dados, identificando de qual arquivo cada registro foi carregado e quando a ingestão foi realizada.
- **Processo idempotente**: a execução do processo mais de uma vez não deve gerar registros duplicados.

## Objetivo

Transformar os arquivos CSV brutos em uma tabela Delta na camada Bronze, preservando integralmente os dados originais e garantindo rastreabilidade, reprocessamento seguro e governança dos dados.

## Entrada

```text
/Volumes/voo_bem/bronze/arquivos/vra/

In [0]:
from pyspark.sql import functions as F

CAMINHO = "/Volumes/voo_bem/bronze/arquivos/vra/*.csv"
TABELA = "voo_bem.bronze.vra"

## Leitura dos Arquivos CSV

Os arquivos da ANAC possuem algumas características que exigem configurações específicas para garantir uma ingestão correta na camada Bronze.

### Separador

```python
.option("sep", ";")
```

Os arquivos utilizam ponto e vírgula (`;`) como separador de colunas.

Sem essa configuração, o Spark interpretaria cada linha como uma única coluna.

---

### Ignorar a Primeira Linha

```python
.option("skipRows", 1)
```

A primeira linha contém apenas informações administrativas, como a data de atualização do arquivo.

Essa linha não deve ser tratada como dado nem como cabeçalho.

---

### Utilizar Cabeçalho

```python
.option("header", True)
```

Após ignorar a primeira linha, a próxima linha passa a ser utilizada como nome das colunas.

---

### Inferência de Schema Desligada

```python
.option("inferSchema", False)
```

Na camada Bronze o Spark não deve tentar identificar automaticamente os tipos das colunas.

Todos os campos devem permanecer como:

```text
string
```

Objetivos:

- Preservar fielmente os dados de origem.
- Evitar conversões automáticas.
- Garantir rastreabilidade.
- Permitir reprocessamentos seguros.
- Aplicar regras de qualidade apenas na camada Silver.

Exemplo:

```text
00123
```

deve permanecer:

```text
00123
```

e não ser convertido para:

```text
123
```

---

## Resumo das Configurações

| Configuração | Finalidade |
|-------------|------------|
| `sep=";"` | Utilizar o separador correto dos arquivos |
| `skipRows=1` | Ignorar a linha de atualização do arquivo |
| `header=True` | Utilizar a segunda linha como cabeçalho |
| `inferSchema=False` | Manter todas as colunas como string |

---

## Regra Fundamental da Bronze

A camada Bronze deve armazenar os dados exatamente como foram recebidos da origem.

Não realizamos:

- Tipagem
- Limpeza
- Filtros
- Regras de negócio

A única exceção é a inclusão de colunas de auditoria para rastrear a origem e o momento da ingestão dos dados.

## Leitura dos Arquivos

Nesta etapa, os arquivos CSV da ANAC são carregados para um DataFrame Spark seguindo as regras da camada Bronze.

### Formato dos Arquivos

```python
spark.read.format("csv")
```

Os dados de origem estão armazenados em arquivos CSV.

---

### Separador

```python
.option("sep", ";")
```

Os datasets utilizam ponto e vírgula (`;`) como delimitador de colunas.

Sem essa configuração, o Spark interpretaria toda a linha como uma única coluna.

---

### Cabeçalho

```python
.option("header", "true")
```

Após descartar a primeira linha do arquivo, a próxima linha passa a ser utilizada como cabeçalho.

Os nomes das colunas serão obtidos automaticamente a partir desta linha.

---

### Ignorar Primeira Linha

```python
.option("skipRows", 1)
```

A primeira linha contém apenas informações administrativas do arquivo, como:

```text
Atualizado em: ...
```

Essa linha não representa dados válidos e não deve ser carregada para a camada Bronze.

Além disso, essa configuração ajuda a eliminar possíveis caracteres BOM (*Byte Order Mark*) presentes no início do arquivo, evitando problemas na leitura do cabeçalho.

---

### Desabilitar Tratamento de Aspas

```python
.option("quote", "")
```

Evita que o Spark interprete aspas como delimitadores especiais de texto.

Essa configuração ajuda a preservar os dados exatamente como foram recebidos da origem.

---

### Desabilitar Escape

```python
.option("escape", "")
```

Evita tratamento automático de caracteres de escape durante a leitura dos arquivos.

O objetivo é manter a máxima fidelidade possível ao conteúdo original.

--

In [0]:
bruto = (
    spark.read.format("csv")
    .option("sep", ";")
    .option("header", "true")
    .option("skipRows", 1)
    .option("quote", "")
    .option("escape", "")
    .option("encoding", "UTF-8")
    .option("mode", "PERMISSIVE")
    .load(CAMINHO)
)

print("colunas lidas do arquivo:")

for c in bruto.columns:
    print(f" - {c!r}")

%md
### Validação do Cabeçalho

Após a leitura dos arquivos, foi realizada uma inspeção das colunas identificadas pelo Spark.

Objetivos da validação:

- Confirmar que o cabeçalho foi interpretado corretamente.
- Verificar a remoção da linha "Atualizado em...".
- Confirmar a ausência de caracteres BOM.
- Garantir que todas as colunas esperadas foram carregadas.
- Validar a estrutura antes da criação da tabela Bronze.

Resultado:

✅ Arquivos carregados com sucesso.

✅ Cabeçalho interpretado corretamente.

✅ Colunas identificadas conforme esperado.

✅ Estrutura pronta para as próximas etapas da ingestão.
``

%md
## Nomes de Coluna: o Delta não aceita espaços

### Problema

Os arquivos CSV da ANAC possuem nomes de colunas como:

```text
ICAO Empresa Aérea
Número Voo
Código Autorização (DI)
```

Esses nomes são válidos em arquivos CSV, porém podem ser inválidos em tabelas Delta devido à presença de caracteres especiais e espaços.

Exemplos de caracteres que podem gerar problemas:

```text
espaço
.
;
:
(
)
[
]
{
}
=
```

---

### Solução

Antes de salvar os dados em Delta, os nomes das colunas são normalizados para um padrão compatível.

Exemplo:

```text
ICAO Empresa Aérea
```

torna-se:

```text
icao_empresa_aerea
```

---

### Isso não viola a regra da Bronze

A camada Bronze preserva:

- o valor dos dados;
- a granularidade dos registros;
- a integridade das informações.

A Bronze **não precisa preservar a grafia exata do cabeçalho** do arquivo original.

Portanto, alterar nomes das colunas por motivos técnicos não é considerado uma transformação de negócio.

---

### O que não muda

Os dados continuam exatamente os mesmos.

Exemplo:

```text
Gol
AZUL
LATAM
```

permanece:

```text
Gol
AZUL
LATAM
```

Nenhuma linha é:

- removida;
- filtrada;
- agregada;
- convertida;
- corrigida.

Apenas os nomes das colunas são adaptados para compatibilidade com o Delta Lake.

---

### Mapeamento Explícito

A correspondência entre o nome original e o nome normalizado deve ser explícita no código.

Exemplo:

```text
"ICAO Empresa Aérea"
        ↓
icao_empresa_aerea
```

Evita-se o uso de transformações automáticas ou expressões regulares genéricas (`regexp_replace`) que dificultem a auditoria.

O objetivo é que seja possível identificar com clareza a origem de cada coluna e manter a rastreabilidade entre o arquivo CSV original e a tabela Delta criada na camada Bronze.

---

###

In [0]:
bruto.columns

**Limpar a linha das colunas **

In [0]:
bruto = bruto.toDF(
    *[c.replace('"', '') for c in bruto.columns]
)

print(bruto.columns)

In [0]:
RENOMEAR = {
    "ICAO Empresa Aérea": "icao_empresa",
    "Número Voo": "numero_voo",
    "Código Autorização (DI)": "codigo_di",
    "Código Tipo Linha": "codigo_tipo_linha",
    "ICAO Aeródromo Origem": "icao_origem",
    "ICAO Aeródromo Destino": "icao_destino",
    "Partida Prevista": "partida_prevista",
    "Partida Real": "partida_real",
    "Chegada Prevista": "chegada_prevista",
    "Chegada Real": "chegada_real",
    "Situação Voo": "situacao_voo",
    "Código Justificativa": "codigo_justificativa",
}

faltando = [c for c in RENOMEAR if c not in bruto.columns]

assert not faltando, (
    f"Coluna esperada não encontrada no CSV: {faltando}"
)

renomeado = bruto.select(
    *[
        F.col(f"`{origem}`")
         .cast("string")
         .alias(novo)
        for origem, novo in RENOMEAR.items()
    ]
)

%md
## Normalização das Colunas

Após validar o layout do arquivo, os nomes das colunas são convertidos para um padrão compatível com Delta Lake.

### Exemplo

```text
ICAO Empresa Aérea
```

↓

```text
icao_empresa
```

### Validação do Contrato

Antes da renomeação, é realizada uma validação para garantir que todas as colunas esperadas estão presentes no arquivo.

```python
faltando = [c for c in RENOMEAR if c not in bruto.columns]
```

Caso alguma coluna esteja ausente, o pipeline interrompe a execução.

### Padronização

As colunas são renomeadas utilizando um dicionário explícito de mapeamento.

Benefícios:

- Compatibilidade com Delta Lake.
- Maior legibilidade.
- Facilidade para consultas SQL.
- Rastreabilidade entre origem e destino.
- Governança dos dados.

### Resultado

Foi criado o DataFrame `renomeado` contendo os mesmos dados do CSV original, porém com nomes de colunas padronizados e compatíveis com o ambiente Lakehouse.

%md
## Auditoria

A camada Bronze adiciona colunas técnicas de auditoria para garantir rastreabilidade dos dados.

Essas colunas não existem no arquivo CSV original e são geradas durante a ingestão.

---

### __arquivo_origem

Armazena o caminho do arquivo responsável pela origem do registro.

Exemplo:

```text
/Volumes/voo_bem/bronze/arquivos/vra/VRA_202510.csv
```

Objetivos:

- identificar a origem de cada registro;
- facilitar auditorias;
- simplificar reprocessamentos;
- auxiliar investigações de inconsistências.

---

### __ingerido_em

Armazena a data e hora da ingestão dos dados.

Exemplo:

```text
2026-09-15 18:42:15
```

Objetivos:

- registrar quando a carga foi executada;
- permitir monitoramento do pipeline;
- facilitar rastreabilidade temporal.

---

### Benefícios da Auditoria

- Governança de dados.
- Rastreabilidade.
- Monitoramento.
- Reprocessamento seguro.
- Investigação de problemas.

---

### Importante

A inclusão de colunas de auditoria não altera os dados de negócio.

Os valores originais permanecem inalterados, respeitando os princípios da camada Bronze da Arquitetura Medallion.
`

In [0]:
bronze = (
    renomeado.withColumn(
        "__arquivo_origem",
        F.col("_metadata.file_name")
    )
    .withColumn(
        "__ingerido_em",
        F.current_timestamp()
    )
)

%md
## Colunas de Auditoria

Após a normalização dos nomes das colunas, são adicionadas informações técnicas de auditoria.

Essas informações não existem nos arquivos CSV originais, mas são fundamentais para rastreabilidade dos dados.

---

### __arquivo_origem

```python
F.col("_metadata.file_name")
```

Armazena o nome do arquivo responsável pela origem do registro.

Exemplo:

```text
VRA_202510.csv
VRA_202511.csv
```

Benefícios:

- Identificação da origem dos dados.
- Facilidade para auditorias.
- Suporte a reprocessamentos.
- Investigação de inconsistências.

---

### __ingerido_em

```python
F.current_timestamp()
```

Armazena a data e hora da ingestão dos dados.

Exemplo:

```text
2026-09-15 19:10:35
```

Benefícios:

- Rastreabilidade temporal.
- Monitoramento de cargas.
- Controle de execução de pipelines.
- Governança de dados.

---

### Importante

As colunas de auditoria não alteram os dados de negócio.

Os registros continuam preservando integralmente os valores originais recebidos da ANAC, respeitando os princípios da camada Bronze da Arquitetura Medallion.

### Resultado

Foi criado o DataFrame `bronze`, contendo:

- Colunas padronizadas.
- Dados originais preservados.
- Informações de auditoria.
- Estrutura pronta para persistência em Delta Lake.

%md
# Escrita Idempotente

## O que é Idempotência?

Um processo é considerado idempotente quando pode ser executado várias vezes produzindo sempre o mesmo resultado final.

Exemplo:

```text
Executar hoje
↓
100.000 registros

Executar novamente amanhã
↓
100.000 registros
```

O resultado permanece consistente, sem gerar duplicações.

---

## Estratégia Utilizada

Nesta implementação será utilizado:

```python
mode("overwrite")
```

Ou seja, a cada execução a tabela Bronze será completamente recriada a partir dos arquivos presentes no volume.

Essa abordagem é conhecida como:

```text
Full Refresh Determinístico
```

---

## Por que não utilizar APPEND?

A estratégia de `append` adiciona novos registros à tabela existente.

Exemplo:

```text
1ª execução
100.000 registros

2ª execução
+100.000 registros

Resultado:
200.000 registros
```

Isso pode gerar duplicidades caso os arquivos sejam processados novamente.

---

## Motivo 1: Fonte Imutável e Completa

O volume contém todos os arquivos necessários para reconstruir a tabela.

Exemplo:

```text
VRA_202510.csv
VRA_202511.csv
VRA_202512.csv
...
```

A cada execução, o pipeline lê o conjunto completo de arquivos.

Portanto, o estado final da tabela pode ser totalmente reconstruído a partir da origem.

---

## Motivo 2: Ausência de Chave Natural Única

O dataset VRA não possui uma chave de negócio única capaz de identificar cada registro de forma inequívoca.

Exemplo:

```text
Mesmo voo
Mesmo dia
Mesmo aeroporto
```

podem aparecer em mais de um registro.

Nesse cenário, uma estratégia de deduplicação baseada em chave seria mais complexa e menos confiável.

---

## Benefícios da Estratégia

- Simplicidade operacional.
- Facilidade de manutenção.
- Reprocessamento seguro.
- Ausência de duplicidades.
- Resultado determinístico.
- Maior confiabilidade da camada Bronze.

---

## Resultado Esperado

Independentemente da quantidade de execuções:

```text
Arquivos da origem
        ↓
Full Refresh
        ↓
Tabela Bronze
```

Sempre teremos uma única versão consistente da tabela, representando fielmente o conteúdo dos arquivos presentes no volume.

### Motivo 3: Overwrite no Delta é Atômico

Ao utilizar:

```python
.mode("overwrite")
```

a substituição da tabela acontece de forma atômica.

Isso significa que:

- a nova versão aparece completa;
- ou a versão anterior continua disponível;
- nunca existe um estado intermediário visível para quem consulta a tabela.

Em outras palavras:

```text
Versão antiga
      ↓
Overwrite
      ↓
Versão nova
```

O usuário nunca enxerga uma tabela parcialmente atualizada.

---

### Motivo 4: Histórico Preservado

O uso de `overwrite` não significa perda definitiva de histórico.

No Delta Lake, cada nova gravação gera uma nova versão do log transacional.

Exemplo:

```text
Versão 0
Versão 1
Versão 2
Versão 3
```

As versões anteriores continuam disponíveis através dos recursos de versionamento do Delta Lake.

Isso permite:

- auditoria;
- rastreabilidade;
- recuperação de versões anteriores;
- análises históricas.

---

### O que muda entre duas execuções?

A única coluna que muda naturalmente é:

```text
__ingerido_em
```

pois ela registra o momento exato da carga.

Exemplo:

Execução 1:

```text
2026-09-15 19:10:00
```

Execução 2:

```text
2026-09-15 20:00:00
```

Fora isso, o conjunto de registros permanece exatamente o mesmo, desde que os arquivos de origem não tenham sido alterados.

Esse é justamente o objetivo de uma carga idempotente:

> Executar várias vezes produz o mesmo conjunto de dados, alterando apenas metadados técnicos relacionados à execução.


In [0]:
(
    bronze.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA)
)

print(f"{TABELA}: {spark.table(TABELA).count():,} linhas")

%md
# Persistência da Camada Bronze

Após a leitura, validação, normalização e auditoria dos dados, o DataFrame é persistido como uma tabela Delta na camada Bronze.

---

## Formato Delta Lake

```python
.write.format("delta")
```

Os dados são armazenados utilizando o formato Delta Lake, que oferece:

- Transações ACID.
- Versionamento.
- Time Travel.
- Evolução de schema.
- Melhor desempenho para workloads analíticos.

---

## Escrita Idempotente

```python
.mode("overwrite")
```

A estratégia adotada é Full Refresh Determinístico.

A cada execução:

1. Todos os arquivos são lidos novamente.
2. A tabela é recriada integralmente.
3. Não há risco de duplicação de registros.

---

## Atualização do Schema

```python
.option("overwriteSchema", "true")
```

Permite que alterações na estrutura da tabela sejam refletidas durante a recriação.

---

## Tabela de Destino

```python
.saveAsTable(TABELA)
```

Tabela criada:

```text
voo_bem.bronze.vra
```

---

## Validação da Carga

Após a escrita, é realizada uma contagem dos registros carregados.

```python
spark.table(TABELA).count()
```

Essa validação confirma que:

- A tabela foi criada com sucesso.
- Os dados estão acessíveis.
- A carga foi concluída corretamente.

---

## Resultado

A camada Bronze foi implementada com sucesso contendo:

- Dados originais preservados.
- Colunas padronizadas.
- Colunas de auditoria.
- Persistência em Delta Lake.
- Escrita idempotente.
- Estrutura pronta para evolução nas camadas Silver e Gold.

In [0]:
spark.sql(f"""
COMMENT ON TABLE {TABELA} IS
'Bronze (Voo Regular Ativo) da ANAC.

Dados brutos carregados a partir dos arquivos CSV armazenados em /Volumes/voo_bem/bronze/arquivos/vra/.

Regras aplicadas:
- Todas as colunas armazenadas como string.
- Nenhuma linha descartada.
- Colunas de auditoria adicionadas (__arquivo_origem e __ingerido_em).
- Carga full refresh idempotente utilizando overwrite.

Origem: Dados Abertos ANAC.'
""")

%md
# Documentação da Tabela

Após a criação da tabela Delta, é adicionada uma descrição oficial ao catálogo.

## Objetivo

Documentar:

- origem dos dados;
- regras da camada Bronze;
- estratégia de carga;
- localização da fonte.

## Benefícios

- Governança de dados.
- Facilidade para novos usuários.
- Maior rastreabilidade.
- Melhor documentação do Lakehouse.

## Informações Registradas

- Fonte: ANAC.
- Dataset: Voo Regular Ativo (VRA).
- Camada: Bronze.
- Tipo de carga: Full Refresh.
- Estratégia: Idempotente.
- Localização dos arquivos:
  `/Volumes/voo_bem/bronze/arquivos/vra/`

## Resultado

A tabela fica autoexplicativa dentro do Unity Catalog, permitindo que qualquer usuário compreenda rapidamente sua origem e finalidade.

In [0]:
display(
    spark.sql(f"""
        SELECT
            __arquivo_origem,
            COUNT(*) AS linhas,
            MAX(__ingerido_em) AS ingerido_em
        FROM {TABELA}
        GROUP BY __arquivo_origem
        ORDER BY __arquivo_origem
    """)
)